# 統計学の基礎：母集団・標本・観測値（R版）

このノートブックでは、統計学における3つの重要な概念を視覚的に学びます。

| 概念 | 定義 | 例 |
|------|------|----|
| **母集団 (Population)** | 調査・分析の対象となるすべての個体の集合 | J国全体の成人男性の身長 |
| **標本 (Sample)** | 母集団から抽出した一部の個体の集合 | J国のある地域で調査した500人の身長 |
| **観測値 (Observation)** | 標本の中の各個体から実際に測定した具体的な値 | 175cm, 168cm, 182cm, ... |

---

In [ ]:
# vctrs を先にアップデートしてからセッションを再起動する
install.packages("vctrs", quiet = TRUE)

# ↑ 実行後、メニューの「ランタイム → セッションを再起動」を行ってから
# 以下のセルを実行してください


In [ ]:
# パッケージのインストールと読み込み
# (vctrs更新 & セッション再起動後にこのセルから実行してください)
install.packages(c("ggplot2", "dplyr", "tidyr", "patchwork"), quiet = TRUE)

library(ggplot2)
library(dplyr)
library(tidyr)
library(patchwork)

# 共通テーマ設定
theme_set(
  theme_bw(base_size = 12) +
  theme(
    plot.title   = element_text(face = "bold", size = 13),
    panel.grid.minor = element_blank()
  )
)

set.seed(42)
cat("準備完了!
")


## Step 1: 母集団を作る

「J国の成人男性の身長」を母集団として設定します。

- **母集団の平均 (μ)**: 171 cm
- **母集団の標準偏差 (σ)**: 6 cm
- **母集団のサイズ (N)**: 10,000人

In [ ]:
# ===== 母集団の設定 =====
mu    <- 171      # 母平均 (cm)
sigma <- 6        # 母Std Dev (cm)
N     <- 10000    # 母集団サイズ

# 母集団データを生成（正規分布）
population <- rnorm(N, mean = mu, sd = sigma)

cat(strrep("=", 40), "\n")
cat("母集団 (Population) の情報\n")
cat(strrep("=", 40), "\n")
cat(sprintf("  サイズ (N)       : %s 人\n", format(N, big.mark = ",")))
cat(sprintf("  真の平均 (mu)    : %g cm\n", mu))
cat(sprintf("  真の標準偏差 (sigma) : %g cm\n", sigma))
cat(sprintf("  実際の平均       : %.2f cm\n", mean(population)))
cat(sprintf("  実際の標準偏差   : %.2f cm\n", sd(population)))
cat("\n最初の10人の身長（観測値の例）:\n")
cat(round(head(population, 10), 1), "\n")

In [ ]:
# Visualize population distribution
df_pop <- data.frame(height = population)

ggplot(df_pop, aes(x = height)) +
  geom_histogram(bins = 60, fill = "#4A90D9", alpha = 0.7, color = "white", linewidth = 0.3) +
  geom_vline(xintercept = mean(population), color = "#E74C3C",
             linewidth = 1.2, linetype = "dashed") +
  annotate("label", x = mean(population) + 4, y = Inf, vjust = 1.5,
           label = sprintf("Population mean mu = %g cm", mu),
           color = "#E74C3C", fill = "white", size = 3.5) +
  annotate("label", x = max(population) - 2, y = Inf, vjust = 1.5, hjust = 1,
           label = sprintf("N = %s人\nmu = %gcm\nsigma = %gcm", format(N, big.mark=","), mu, sigma),
           fill = "lightyellow", color = "#E74C3C", size = 3.5) +
  labs(
    title = "Population: Adult Male Height in Country J (N=10,000)",
    x = "Height (cm)", y = "Count"
  )

## Step 2: 母集団から標本を抽出する

現実には母集団全員を調査することは困難です。そこで母集団から一部を**無作為抽出（ランダムサンプリング）**します。

抽出した各データが「**観測値**」、集まり全体が「**標本**」です。

In [ ]:
# ===== 標本の抽出 =====
n <- 50   # 標本サイズ

# Random sampling（非復元）
sample_idx <- sample(N, n, replace = FALSE)
samp       <- population[sample_idx]

cat(strrep("=", 40), "\n")
cat("標本 (Sample) の情報\n")
cat(strrep("=", 40), "\n")
cat(sprintf("  標本サイズ (n)   : %d 人\n", n))
cat(sprintf("  標本平均 (x-bar) : %.2f cm  <- Pop. mean mu=%gcm の推定値\n", mean(samp), mu))
cat(sprintf("  標本標準偏差 (s) : %.2f cm  <- 母標準偏差 sigma=%gcm の推定値\n", sd(samp), sigma))
cat("\n各観測値:\n")
# 5列で表示
vals <- sprintf("[%2d] %.1fcm", seq_along(samp), samp)
matrix(vals, ncol = 5, byrow = FALSE) |> apply(1, paste, collapse = "  ") |> cat(sep = "\n")

In [ ]:
# Compare population and sample
df_samp <- data.frame(height = samp)

p_pop <- ggplot(df_pop, aes(x = height)) +
  geom_histogram(bins = 50, fill = "#4A90D9", alpha = 0.6, color = "white") +
  geom_vline(xintercept = mean(population), color = "#E74C3C",
             linewidth = 1.2, linetype = "dashed") +
  annotate("text", x = mean(population) + 5, y = Inf, vjust = 2,
           label = sprintf("Pop. mean mu = %.1fcm", mean(population)),
           color = "#E74C3C", size = 3.5) +
  labs(title = "Population  N = 10,000", x = "Height (cm)", y = "Count")

p_samp <- ggplot(df_samp, aes(x = height)) +
  geom_histogram(bins = 15, fill = "#E67E22", alpha = 0.7, color = "white") +
  geom_vline(xintercept = mean(samp), color = "#E74C3C",
             linewidth = 1.2, linetype = "dashed") +
  geom_vline(xintercept = mu, color = "#4A90D9",
             linewidth = 1, linetype = "dotted", alpha = 0.8) +
  annotate("text", x = mean(samp) + 4, y = Inf, vjust = 2,
           label = sprintf("Sample mean x-bar = %.1fcm", mean(samp)),
           color = "#E74C3C", size = 3.5) +
  annotate("text", x = mu - 5, y = Inf, vjust = 3.5,
           label = sprintf("True pop. mean mu = %gcm", mu),
           color = "#4A90D9", size = 3.5) +
  labs(title = sprintf("Sample  n = %d (random sampling)", n), x = "Height (cm)", y = "Count")

p_pop + p_samp +
  plot_annotation(title = "Population vs Sample",
                  theme = theme(plot.title = element_text(face = "bold", size = 14)))

## Step 3: 観測値を詳しく見る

標本中の**各測定値が「観測値」**です。観測値から統計量（平均・標準偏差など）を計算し、母集団のパラメータを推定します。

In [ ]:
# ① ドットプロット
df_samp$jitter_y <- runif(n, -0.3, 0.3)

p_dot <- ggplot(df_samp, aes(x = height, y = jitter_y, color = height)) +
  geom_point(size = 3, alpha = 0.8) +
  geom_vline(xintercept = mean(samp), color = "red",
             linewidth = 1.2, linetype = "dashed") +
  scale_color_distiller(palette = "RdYlBu") +
  annotate("text", x = mean(samp) + 3, y = 0.35,
           label = sprintf("x-bar = %.1fcm", mean(samp)),
           color = "red", size = 3.5) +
  labs(title = "Dot plot of observations", x = "Height (cm)", y = "") +
  theme(axis.text.y = element_blank(), axis.ticks.y = element_blank(),
        legend.position = "bottom", legend.key.width = unit(1.5, "cm")) +
  guides(color = guide_colorbar(title = "Height (cm)"))

# ② 箱ひげ図
p_box <- ggplot(df_samp, aes(y = height)) +
  geom_boxplot(fill = "#E67E22", alpha = 0.6, width = 0.4,
               outlier.color = "red", outlier.size = 2) +
  geom_hline(yintercept = mu, color = "#4A90D9",
             linewidth = 1, linetype = "dotted") +
  annotate("text", x = 0.35, y = mu + 1,
           label = sprintf("Pop. mean mu=%gcm", mu),
           color = "#4A90D9", size = 3.5) +
  labs(title = "Boxplot (Sample)", y = "Height (cm)") +
  theme(axis.text.x = element_blank(), axis.ticks.x = element_blank())

# ③ Parameters vs Sample Statistics
df_compare <- data.frame(
  stat    = rep(c("Mean (cm)", "Std Dev (cm)"), each = 2),
  type    = rep(c("Parameter (true)", "Sample statistic (estimate)"), 2),
  value   = c(mu, mean(samp), sigma, sd(samp))
)

p_bar <- ggplot(df_compare, aes(x = stat, y = value, fill = type)) +
  geom_col(position = "dodge", alpha = 0.85, width = 0.6) +
  geom_text(aes(label = sprintf("%.2f", value)),
            position = position_dodge(width = 0.6),
            vjust = -0.5, size = 3.5, fontface = "bold") +
  scale_fill_manual(values = c("Parameter (true)" = "#4A90D9",
                               "Sample statistic (estimate)" = "#E67E22")) +
  labs(title = "Parameters vs Sample Statistics", x = "", y = "", fill = "") +
  theme(legend.position = "bottom")

(p_dot | p_box | p_bar) +
  plot_annotation(title = "Analysis of Observations",
                  theme = theme(plot.title = element_text(face = "bold", size = 14)))

## Step 4: 標本サイズの影響を理解する

標本サイズ `n` が大きいほど、標本平均は母平均に近づきます（**大数の法則**）。

複数の標本サイズで繰り返し抽出してみましょう。

In [ ]:
# Estimation accuracy by sample size
sample_sizes <- c(5, 10, 30, 50, 100, 500)
n_trials     <- 1000

# 各サイズで1000回抽出して標本平均を記録
results <- lapply(sample_sizes, function(sz) {
  means <- replicate(n_trials, mean(sample(population, sz, replace = FALSE)))
  data.frame(n_size = sz, sample_mean = means)
})
df_results <- do.call(rbind, results)
df_results$n_label <- factor(
  sprintf("n = %d", df_results$n_size),
  levels = sprintf("n = %d", sample_sizes)
)

# 標準誤差ラベル
se_labels <- df_results |>
  group_by(n_label) |>
  summarise(se = sd(sample_mean), .groups = "drop") |>
  mutate(label = sprintf("SE = %.2fcm", se))

ggplot(df_results, aes(x = sample_mean)) +
  geom_histogram(aes(fill = n_label), bins = 40, alpha = 0.75,
                 color = "white", linewidth = 0.2, show.legend = FALSE) +
  geom_vline(xintercept = mu, color = "red", linewidth = 0.8, linetype = "dashed") +
  geom_text(data = se_labels, aes(label = label),
            x = Inf, y = Inf, hjust = 1.1, vjust = 1.5, size = 3.5, fontface = "bold") +
  facet_wrap(~n_label, nrow = 2) +
  scale_x_continuous(limits = c(155, 187)) +
  scale_fill_viridis_d() +
  labs(
    title = sprintf("Distribution of sample means (%d trials each)", n_trials),
    subtitle = "Larger n -> narrower distribution -> higher accuracy",
    x = "Sample mean (cm)", y = "Frequency"
  )

In [ ]:
# 標準誤差と標本サイズの関係
se_df <- df_results |>
  group_by(n_size) |>
  summarise(se_empirical = sd(sample_mean), .groups = "drop") |>
  mutate(se_theoretical = sigma / sqrt(n_size))

se_long <- se_df |>
  pivot_longer(cols = c(se_empirical, se_theoretical),
               names_to = "type", values_to = "se") |>
  mutate(type = ifelse(type == "se_empirical", "Empirical SE",
                       sprintf("理論値 sigma/sqrt(n) (sigma=%g)", sigma)))

p_se <- ggplot(se_long, aes(x = n_size, y = se, color = type, shape = type)) +
  geom_line(linewidth = 1.2) +
  geom_point(size = 3) +
  geom_text(data = filter(se_long, type == "Empirical SE"),
            aes(label = sprintf("SE=%.2f", se)),
            nudge_x = 15, nudge_y = 0.05, size = 3, color = "#E67E22") +
  scale_color_manual(values = c("Empirical SE" = "#E67E22",
                                 "Theoretical sigma/sqrt(n) (sigma=6)" = "#4A90D9")) +
  labs(title = "Sample size vs Standard Error",
       x = "Sample size n", y = "Standard Error SE (cm)",
       color = "", shape = "") +
  theme(legend.position = "bottom")

# 概念図（母集団→標本→観測値）
concept_data <- data.frame(
  x     = c(5, 5, 5),
  y     = c(7.5, 4, 1.2),
  label = c(sprintf("母集団\nN=%s人\nmu=%gcm, sigma=%gcm", format(N, big.mark=","), mu, sigma),
             sprintf("標本\nn=%d人", n),
             "観測値\n175.2, 168.7, 182.1, ..."),
  fill  = c("#AED6F1", "#FAD7A0", "#D5F5E3"),
  col   = c("#2E86C1", "#E67E22", "#1E8449")
)

p_concept <- ggplot() +
  geom_point(data = concept_data,
             aes(x = x, y = y, size = c(28, 18, 14)),
             color = concept_data$col, fill = concept_data$fill,
             shape = 21, show.legend = FALSE) +
  geom_text(data = concept_data, aes(x = x, y = y, label = label),
            size = 3, lineheight = 1.3) +
  annotate("segment", x = 5, xend = 5, y = 6.2, yend = 5.2,
           arrow = arrow(length = unit(0.3, "cm")), color = "#E74C3C", linewidth = 1) +
  annotate("text", x = 5.4, y = 5.7, label = "Random sampling",
           color = "#E74C3C", size = 3.2, fontface = "bold") +
  annotate("segment", x = 5, xend = 5, y = 3.0, yend = 2.1,
           arrow = arrow(length = unit(0.3, "cm")), color = "#27AE60", linewidth = 1) +
  annotate("text", x = 5.4, y = 2.55, label = "Measure each",
           color = "#27AE60", size = 3.2, fontface = "bold") +
  scale_size_identity() +
  xlim(3, 7.5) + ylim(0, 10) +
  labs(title = "Conceptual diagram") +
  theme_void(base_size = 12) +
  theme(plot.title = element_text(face = "bold", size = 13, hjust = 0.5))

(p_se | p_concept) +
  plot_annotation(title = "SE and the relationship among Population / Sample / Observations",
                  theme = theme(plot.title = element_text(face = "bold", size = 14)))


## Step 5: インタラクティブ体験 - 自分で標本を抽出してみよう

標本サイズを変えて、標本平均がどのくらい母平均に近くなるか確認しましょう。

In [ ]:
# ===== 自分で試してみよう！=====
# 下の数値を変えて実行してみてください

MY_SAMPLE_SIZE <- 30   # <- 標本サイズを変えてみよう（5〜5000の間）
MY_SEED        <- 99   # <- 乱数シードを変えると別の標本が得られる

# =================================
set.seed(MY_SEED)
my_samp <- sample(population, MY_SAMPLE_SIZE, replace = FALSE)
df_my   <- data.frame(height = my_samp)

# 標本ヒストグラム
p_my <- ggplot(df_my, aes(x = height)) +
  geom_histogram(bins = max(5, MY_SAMPLE_SIZE %/% 5),
                 fill = "#9B59B6", alpha = 0.7, color = "white") +
  geom_vline(xintercept = mean(my_samp), color = "red",
             linewidth = 1.3, linetype = "dashed") +
  geom_vline(xintercept = mu, color = "blue",
             linewidth = 1, linetype = "dotted", alpha = 0.8) +
  annotate("text", x = mean(my_samp) + 3, y = Inf, vjust = 2,
           label = sprintf("Sample mean x-bar = %.2fcm", mean(my_samp)),
           color = "red", size = 3.5) +
  annotate("text", x = mu - 4, y = Inf, vjust = 3.5,
           label = sprintf("True pop. mean mu = %gcm", mu),
           color = "blue", size = 3.5) +
  labs(title = sprintf("Your sample (n=%d)", MY_SAMPLE_SIZE),
       x = "Height (cm)", y = "Count")

print(p_my)

# 結果サマリー
error          <- abs(mean(my_samp) - mu)
error_pct      <- error / mu * 100
se_theoretical <- sigma / sqrt(MY_SAMPLE_SIZE)

cat(strrep("=", 45), "\n")
cat("結果サマリー\n")
cat(strrep("=", 45), "\n")
cat(sprintf("  母集団の真の平均 (mu)      : %.2f cm\n", mu))
cat(sprintf("  あなたの標本平均 (x-bar)   : %.2f cm\n", mean(my_samp)))
cat(sprintf("  推定誤差 |x-bar - mu|      : %.2f cm (%.1f%%)\n", error, error_pct))
cat(sprintf("  理論的な標準誤差 sigma/sqrt(n): %.2f cm\n", se_theoretical))
cat(sprintf("  標本の標準偏差 (s)         : %.2f cm\n", sd(my_samp)))
cat(sprintf("  標本サイズ (n)             : %d 人\n", MY_SAMPLE_SIZE))
cat(strrep("-", 45), "\n")

if (error <= se_theoretical) {
  cat("優秀！標準誤差以内に収まっています\n")
} else if (error <= 2 * se_theoretical) {
  cat("良好！2x標準誤差以内に収まっています\n")
} else {
  cat("今回は誤差が大きめです（運が悪かった）\n")
}

cat("\nヒント: MY_SAMPLE_SIZE を大きくすると推定誤差が小さくなります！\n")
cat("　　　  MY_SEED を変えると別の標本が抽出されます。\n")

## まとめ

In [ ]:
cat(strrep("=", 55), "\n")
cat("  統計学の基礎概念まとめ\n")
cat(strrep("=", 55), "\n\n")
cat("+", strrep("-", 51), "+\n", sep="")
cat("| 概念          記号   意味                        |\n")
cat("+", strrep("-", 51), "+\n", sep="")
cat("| 母集団        N      調査対象の全体集合           |\n")
cat("| 母平均        mu     母集団全体の平均値           |\n")
cat("| 母標準偏差    sigma  母集団全体のばらつき         |\n")
cat("+", strrep("-", 51), "+\n", sep="")
cat("| 標本          n      母集団から抽出した部分集合   |\n")
cat("| 標本平均      x-bar  標本の平均値（muの推定値）   |\n")
cat("| 標本標準偏差  s      標本のばらつき（sigmaの推定値）|\n")
cat("+", strrep("-", 51), "+\n", sep="")
cat("| 観測値        xi     各個体の測定値              |\n")
cat("| 標準誤差      SE     x-barのばらつき = sigma/sqrt(n)|\n")
cat("+", strrep("-", 51), "+\n\n", sep="")
cat("重要なポイント:\n")
cat("  1. Sample size n が大きいほど、推定精度が上がる\n")
cat("  2. Random samplingにより、偏りのない推定ができる\n")
cat("  3. 標準誤差 SE = sigma/sqrt(n)（nの平方根に反比例）\n")
cat("  4. 観測値の集まりが標本、全体が母集団\n")